In [ ]:
import requests
import json
import time
import pickle
import pandas as pd
from pandas import Timestamp
import numpy as np
from tqdm import tqdm
import os
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
import time
from selenium.webdriver.common.by import By



In [ ]:
option = webdriver.ChromeOptions()
browser  = webdriver.Chrome(ChromeDriverManager().install())

In [ ]:

def scrapeData(contractAddress,token_id,t=.5):
    url = f"https://eth-mainnet.g.alchemy.com/nft/v2/demo/getNFTMetadata?contractAddress={contractAddress}&tokenId={token_id}&refreshCache=false"

    headers = {"accept": "application/json"}

    response = requests.get(url, headers=headers)

    response=json.loads(response.text)
    time.sleep(t)
    return response

def getSmartContractAddress(browser,collection_name):
    try:
        collection_name=collection_name.lower()

        browser.get(f"https://opensea.io/collection/{collection_name}")

        time.sleep(1)

        elements=browser.find_elements(By.XPATH,"/html/body/div[1]/div/main/div/div/div/div[5]/div/div[3]/div[3]/div[3]/div[3]/div/div[1]/article/a")
        smart_contract_address = [elem.get_attribute('href') for elem in elements][0]
        smart_contract_address=smart_contract_address.split("/")[-2]
        time.sleep(1)
    except:
        return None
    return smart_contract_address


In [ ]:
path = "/home/user/Progetti/NFT/BASELINE_FOR_ABLATION"

nft2emb = pickle.load(open(f"../embeddings/nft2emb_dummyViT-train.pkl", 'rb'))


In [ ]:
coll2nfts={}
for nft in nft2emb:
    if eval(nft)[0] not in coll2nfts:
        coll2nfts[eval(nft)[0]]=[]
    coll2nfts[eval(nft)[0]].append(nft)
coll2size={coll:len(coll2nfts[coll]) for coll in coll2nfts}    

df=pd.DataFrame(coll2size.items(),columns=["coll","count"])
df=df.sort_values(by="count",ascending=False)
df["p"]=df["count"]/df["count"].sum()

df.head(10)

In [ ]:
if not os.path.exists("../input/collection2smartcontract.pkl"):
    collection2contractAddress={
    coll:getSmartContractAddress(browser,coll) for coll in tqdm(collections)
    }
    pickle.dump(collection2contractAddress,open("../input/collection2smartcontract.pkl","wb"))
collection2contractAddress=pickle.load(open("../input/collection2smartcontract.pkl","rb"))

In [ ]:
print("Before pruning",len(collection2contractAddress))
collection2contractAddress={
    coll:address for coll,address in collection2contractAddress.items() if address is not None
}
print("After pruning",len(collection2contractAddress))


In [ ]:
nft2scrape=[eval(nft) for nft in nft2emb.keys()]

nft2scrape=[(coll,id) for [coll,id] in nft2scrape if coll in collection2contractAddress]

print("NFT che è possibile sscaricare poichè appartenenti a collection di cui si conosce l'indirizzo dello smart contract ",len(nft2scrape))

In [ ]:
if not os.path.exists("../input/scrapedNFT_minting_time.pkl"):
    data={}
    for coll,id in tqdm(nft2scrape):
        try:
            data_scraped=scrapeData(collection2contractAddress[coll],id)
            data[f'{(coll,id)}']=data_scraped
        except Exception as e:
            print(e)
    pickle.dump(data,open("../input/scrapedNFT_minting_time.pkl","wb"))
    
data=pickle.load(open("../input/scrapedNFT_minting_time.pkl","rb"))

In [ ]:
print("Numero di NFT di cui sono stati ottenuti i metadati",len(data))

In [ ]:
nft2txs = pickle.load(open(f"../input/nft2txs.pkl", "rb"))

In [ ]:
deltas={}
for id in data:
        if eval(id)[0]=="Cryptokitties":
            try:
                dt=(Timestamp(nft2txs[id]["txs"][0]["timestamp"],tz="UTC")-Timestamp(data[id]["metadata"]["created_at"],tz="UTC"))
                deltas[id]=(dt.days*86400+dt.seconds)//86400
            except Exception as e:
                    # print(data[id])
                    pass

        if  eval(id)[0]=="Sorare":  
            try:
                dt=data[id]["metadata"]["attributes"]
                for d in dt:
                    if "display_type" in d and d["display_type"]=="date":
                        dt=d["value"]
                dt=(Timestamp(nft2txs[id]["txs"][0]["timestamp"],tz="UTC")-Timestamp(dt*1e9,tz="UTC"))
                deltas[id]=(dt.days*86400+dt.seconds)//86400
            except Exception as e:
                    pass


                
print("Numero di NFT per cui non è stato possibile trovare il tag created_at",len(data)-len(deltas),"Trovati",len(deltas))

In [ ]:
coll2nfts={}
for nft in deltas:
    if eval(nft)[0] not in coll2nfts:
        coll2nfts[eval(nft)[0]]=[]
    coll2nfts[eval(nft)[0]].append(nft)
coll2size={coll:len(coll2nfts[coll]) for coll in coll2nfts}    

df=pd.DataFrame(coll2size.items(),columns=["coll","count"])
df=df.sort_values(by="count",ascending=False)
df


In [ ]:
import pandas as pd
df=pd.DataFrame().from_dict({"deltas":deltas, "collection":{id:eval(id)[0] for id in deltas}})

print(df.describe())
values, counts = np.unique(df["deltas"], return_counts=True)

print("MODA     :",values[np.argmax(counts)])
print("MEDIANA  :",df["deltas"][len(df["deltas"])//2])
print("MEDIA    :",df["deltas"].mean())
print("STD      :",df["deltas"].std())
print("MIN      :",df["deltas"].min())
print("MAX      :",df["deltas"].max())


In [ ]:
df=df[df["collection"]!="Sorare"]
print(df.describe())
values, counts = np.unique(df["deltas"], return_counts=True)

print("MODA     :",values[np.argmax(counts)])
print("MEDIANA  :",df["deltas"][len(df["deltas"])//2])
print("MEDIA    :",df["deltas"].mean())
print("STD      :",df["deltas"].std())
print("MIN      :",df["deltas"].min())
print("MAX      :",df["deltas"].max())

In [ ]:
import matplotlib.pyplot as plt
values, counts = np.unique(df["deltas"], return_counts=True)
plt.figure(figsize=(10,5))
plt.bar(values,np.log10(counts))
plt.xlabel("Delta")
plt.ylabel("Log10(#occorrenze)")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(15,5))
sns.violinplot(data=df,x="deltas", y="collection",cut=0)

In [ ]:
df["deltas"][df["deltas"]<df["deltas"].mean()].plot(kind="kde")